In [ ]:
from sctoolbox.utils.jupyter import bgcolor, _compare_version

# change the background of input cells
bgcolor("PowderBlue", select=[2, 5, 8, 13, 16, 20, 24, 30, 32, 34, 38, 42])

nb_name = "Palantir.ipynb"

_compare_version(nb_name)

# Trajectory and Pseudotime Analysis
<hr style="border:2px solid black"> </hr>

## 1 - Description

Palantir is an algorithm for reconstructing cellular differentiation trajectories from transcriptomic data. It models cell fate as a probabilistic process, generating a high-resolution pseudo-time ordering of cells, assigning probabilities for differentiation into terminal states, and using entropy to describe cell plasticity. This approach captures both the continuous and uncertain nature of cell state transitions. The workflow in this notebook follows the methodology described in the [official Palantir documentation](https://palantir.readthedocs.io/en/latest/).

**Reference:** Setty, M. *et al.* (2019). *Nature Biotechnology*, 37, 451–460. [https://doi.org/10.1038/s41587-019-0068-4](https://doi.org/10.1038/s41587-019-0068-4)

___

## 2 - Loading packages

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import palantir
import sctoolbox.plotting as pl
import sctoolbox.tools as tools
import sctoolbox.utils as utils
from sctoolbox import settings

settings.settings_from_config("config.yaml", key="Palantir")

with pd.option_context("display.max.rows", None, "display.max_colwidth", None):
    display(utils.general.get_version_report(report="versions.yml"))

___

## 3 - Input/output settings

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
last_notebook_adata = "anndata_marker.h5ad"

___

## 4 - Load anndata

In [ ]:
adata = utils.adata.load_h5ad(last_notebook_adata)

with pd.option_context("display.max.rows", 5, "display.max.columns", None):
    display(adata)
    display(adata.obs)
    display(adata.var)

___

## 5 - Checkpoint and Processing 
<hr style="border:2px solid black"> </hr>

This section verifies that all required preprocessing steps have been correctly applied before starting the Palantir workflow.

To ensure a standardized and reproducible analysis, **Notebooks 1–4 must be completed in full**.  
From the **Group_Markers** notebook, only the marker gene table is required at this stage.

Before proceeding, the dataset is validated using:

In [ ]:
tools.palantir_analysis.is_preprocessed(adata)

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# If you want to exclude certain layers, e.g., "raw" and "MAGIC_imputed_data":
exclude_layers = []

In [ ]:
fig, axs = tools.palantir_analysis.compare_all_layers(adata, exclude_layers=exclude_layers)
plt.show()

In [ ]:
pl.embedding.plot_pca_variance(adata)
plt.show()

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Number of diffusion components computed by Palantir.
# Higher values keep more diffusion structure, but also add more dimensions.
n_components = 10

diffusion_knn = 30  # Number of nearest neighbors used to build the Palantir kernel.

# Backend used for neighbor search in Palantir.
# "scanpy" = recommended default for this workflow
# "sklearn" = exact kNN, mainly useful for testing or comparisons
kernel_backend = "scanpy"

n_steps = 3  # Number of diffusion steps used by MAGIC for smoothing/imputation.

# Number of parallel jobs used by MAGIC.
# Reduce this if your system shows instability or fork/thread warnings.
n_jobs = 3

# False = reuse cached results if available
# True = ignore cache and recompute diffusion maps and MAGIC
recompute_magic = False

# Column in adata.obs that has the cell groups or cell types
clusters_to_analyze = "leiden_0.1"  # Examples: "leiden_0.2", "leiden_0.3" or your cell type names

# Optional: cell groups to remove from the analysis
labels_to_remove = []  # or None

# Output file for the processed AnnData object with diffusion results and MAGIC layer.
save_path = f"../adatas/processed_{last_notebook_adata}_magic.h5ad"

**Warning:**
This step only creates a filtered subset by removing selected cell labels from the current `AnnData` object.
It does **not** recompute PCA, neighbors, UMAP, or any other previously learned representation.

This means that the removed cells are no longer included in the subsequent Palantir analysis itself, but the underlying representation used as input especially `X_pca` as well as any embedding or neighborhood structure computed earlier was originally learned on the full dataset before filtering.

As a result, this step should be interpreted as a **sensitivity/control check only**, not as a fully reprocessed analysis strategy.
For a biologically consistent final workflow, filtering should be followed by recomputation of PCA, neighbors, and UMAP on the filtered subset before running Palantir.

In [ ]:
adata = tools.palantir_analysis.remove_cells_by_label(adata, column_name=clusters_to_analyze, labels_to_remove=labels_to_remove)

**Diffusion maps, multiscale space, and MAGIC imputation**

In this step, Palantir first computes diffusion maps from the existing low-dimensional representation and neighborhood structure. Based on these results, a multiscale diffusion space is derived and stored for downstream trajectory inference. Finally, MAGIC is applied to smooth gene expression across transcriptionally similar cells.

In the processed AnnData object, diffusion-related results are stored under Palantir-specific keys, including `DM_Kernel`, `DM_Similarity`, `DM_EigenValues`, `DM_EigenVectors`, and `DM_EigenVectors_multiscaled`, while the imputed expression matrix is stored in `adata.layers["MAGIC_imputed_data"]`.

Because these computations can be time-consuming on large datasets, it is recommended to run this step once, save the processed `.h5ad` file, and reuse it in downstream analyses to avoid repeated computation and improve reproducibility.

In [ ]:
adata = tools.palantir_analysis.run_diffusion_and_magic(
    adata=adata,
    save_path=save_path,
    n_components=n_components,
    n_jobs=n_jobs,
    n_steps=n_steps,
    recompute_magic=recompute_magic,
    kernel_backend=kernel_backend,
    knn=diffusion_knn,
)

This plot shows the UMAP embedding colored by individual diffusion component values.  It is used to inspect whether diffusion components capture smooth, continuous structure across cells rather than random noise. 
Clear gradients can indicate biologically meaningful variation, such as progression or lineage-related structure, but interpretation should be confirmed with cell annotations and downstream Palantir results.

In [ ]:
palantir.plot.plot_diffusion_components(adata)
plt.show()

___

## 6 - Selection and Visualization of grouping for Palantir Analysis <hr style="border:2px solid black"> </hr>

In this section, the relevant grouping in `adata.obs` is selected for downstream Palantir analysis. 
An overview UMAP plot is used to visualize how these groups are distributed in the embedding. 
This visualization serves as orientation for choosing biologically relevant populations for early-cell and terminal-state definition. 
Optionally, selected groups can be excluded from the downstream analysis if they are not relevant for the trajectory of interest.

In [ ]:
adata.obs.columns

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Column in adata.obs that has the cell groups or cell types
clusters_to_analyze = "leiden_0.1"  # Examples: "leiden_0.2", "leiden_0.3" or your cell type names

In [ ]:
pl.embedding.plot_embedding(adata, color=clusters_to_analyze)
plt.show()
adata.obs[clusters_to_analyze].value_counts()

----------

## 7 - Early Cell Identification
<hr style="border:2px solid black"> </hr>

**Selecting the Early Cell for Palantir**

In this step, we define the Early Cell, which serves as the starting point for trajectory inference. Palantir requires an Early Cell to initiate the reconstruction of differentiation pathways. This cell can be selected in two ways:

* **Manual selection:**
  If the exact cell ID is already known, it can be provided directly.

* **Automatic selection:**
  Palantir identifies an early-cell candidate within a specified cell type based on its position at the boundary of diffusion map space. This provides a data-driven starting point, but it should still be checked for biological plausibility.

The choice of the Early Cell is critical because it defines the reference point from which Palantir computes pseudotime and fate probabilities. Therefore, the selected cell should belong to a biologically plausible early or less differentiated population, for example a stem-like or progenitor-like state.

**Reference:** Setty, M. *et al.* (2019). *Nature Biotechnology*, 37, 451–460. [https://doi.org/10.1038/s41587-019-0068-4](https://doi.org/10.1038/s41587-019-0068-4)

In [ ]:
_ = tools.palantir_analysis.plot_interactive_embedding_cell_ids(
    adata,
    color_by=clusters_to_analyze,
    embedding_key="X_umap",
    title="Interactive UMAP for manual terminal-state selection",
)

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Define the cell type used for automatic early cell selection
celltype = "2"

# Manually define the early cell by its cell ID
# If this is set to None, the early cell will be selected automatically
early_cell = None

In [ ]:
# If no early cell is given, select it automatically using Palantir
if early_cell is None:
    early_cell = palantir.utils.early_cell(
        adata,
        celltype=celltype,
        celltype_column=clusters_to_analyze,
    )
    print("Automatically selected early cell:", early_cell)

# If an early cell is given manually, check that it exists in the dataset
else:
    if early_cell not in adata.obs_names:
        raise ValueError(
            f"Provided early_cell '{early_cell}' not found in adata.obs_names."
        )
    print("Using manually provided early cell:", early_cell)

In [ ]:
# Highlight the selected early cell on the UMAP
palantir.plot.highlight_cells_on_umap(
    adata,
    [early_cell],
    embedding_basis="X_umap",
    figsize=(6, 6),
    annotation_offset=0.03,
    s=1,               # size of all cells
    s_highlighted=20,  # size of highlighted early cell
)

# Add title and display the plot
plt.title(f"Early cell identification\nCell ID: {early_cell}")
plt.tight_layout()
plt.show()

___

## 8 - Identifies Terminal States
<hr style="border:2px solid black"> </hr>

Palantir supports two modes for defining terminal states:

- **Manual mode:**  
  Provide a list of cell IDs to use as terminal states.  
  Use this mode when terminal populations are already known, for example well-defined mature cell types.

- **Automatic mode:**  
  If no terminal states are provided, Palantir identifies terminal-state candidates automatically based on the diffusion map and the specified early cell. 
  These states correspond to endpoint-like regions in diffusion space and represent candidate differentiation fates.

Automatically detected terminal states should be interpreted with caution. 
They are data-driven candidates and should be checked for biological plausibility before being treated as final lineage endpoints.

In [ ]:
_ = tools.palantir_analysis.plot_interactive_embedding_cell_ids(
    adata,
    color_by=clusters_to_analyze,
    embedding_key="X_umap",
    title="Interactive UMAP for manual terminal-state selection",
)

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Terminal states.
# Use None or "" for automatic detection, or provide terminal cell IDs manually,
# e.g. ["<ID1>", "<ID2>", ...].
terminal_states = None

# 2D embedding used for plotting terminal states, e.g. "X_umap" or "X_tsne".
embedding_key = "X_umap"

# Number of nearest neighbors used for graph construction.
# Larger values give a smoother, more connected graph.
# Smaller values increase local sensitivity but can be noisier.
terminal_state_knn = 30

# Number of waypoints used to approximate pseudotime distances.
# More waypoints can improve stability/resolution but increase runtime.
num_waypoints = 1200

# Number of parallel jobs.
# -1 uses all available CPU cores.
n_jobs = -1

# Maximum number of iterations for pseudotime refinement.
# Higher values may improve convergence slightly but increase runtime.
max_iterations = 25

# Random seed used in waypoint sampling and related stochastic steps.
# Improves reproducibility, although very small run-to-run differences may still occur.
seed = 42

In [ ]:
terminal_states_for_palantir = tools.palantir_analysis.detect_and_plot_terminal_states(
    adata=adata,
    early_cell=early_cell,
    terminal_states=terminal_states,
    clusters_to_analyze=clusters_to_analyze,
    embedding_key=embedding_key,
    knn=terminal_state_knn,
    num_waypoints=num_waypoints,
    n_jobs=n_jobs,
    max_iterations=max_iterations,
    seed=seed,
)

print("\nTerminal states passed to Palantir:")
print(terminal_states_for_palantir)

___

## 9 - Run Palantir 
<hr style="border:2px solid black"> </hr>

Running Palantir produces three main outputs:

- **Pseudotime** – orders cells along the inferred differentiation trajectory.
- **Terminal state probabilities** – probability of each cell reaching each terminal state.
- **Entropy** – entropy of terminal-state probabilities, used here as a measure of differentiation potential.

For AnnData input, results are stored by default as:

- `adata.obs["palantir_pseudotime"]` → pseudotime
- `adata.obs["palantir_entropy"]` → entropy
- `adata.obsm["palantir_fate_probabilities"]` → terminal state probabilities
- `adata.uns["palantir_waypoints"]` → waypoints

Pseudotime should be interpreted as a relative ordering of cells along the inferred process, not as absolute biological time.

In [ ]:
pr_res = palantir.core.run_palantir(
    adata,
    early_cell=early_cell,
    terminal_states=terminal_states_for_palantir,
    knn=terminal_state_knn,
    num_waypoints=num_waypoints,
    max_iterations=max_iterations,
    n_jobs=n_jobs,
    seed=seed,
)

In [ ]:
fp = adata.obsm["palantir_fate_probabilities"]

top_fates = fp.max().sort_values(ascending=False).index.tolist()

# prefix for titles
temp_cols = [f"fate_prob: {f}" for f in top_fates]
adata.obs[temp_cols] = fp[top_fates].to_numpy()

pl.embedding.plot_embedding(
    adata,
    method="umap",
    color=["palantir_pseudotime", "palantir_entropy", *temp_cols],
    ncols=4,
    show_title=True,
    size=30,
    frameon=False,
)
plt.show()

# cleanup
adata.obs.drop(columns=temp_cols, inplace=True)

___

## 10 - Selecting cells of a specific branch
<hr style="border:2px solid black"> </hr>

The `select_branch_cells` function in Palantir is used to identify cells associated with a specific branch of the inferred trajectory. 
It selects cells based on their fate probabilities together with their pseudotime position, thereby defining branch-specific subsets for downstream analyses such as gene-trend estimation.

**Key parameters:**

- **q (quantile):** Determines the probability threshold for selecting branch-associated cells. Smaller values make the selection more stringent, whereas larger values include a broader set of cells.
- **eps:** A small tolerance term that slightly relaxes the threshold and can make branch selection more stable.

This branch selection is a downstream analytical filtering step based on the inferred Palantir results. 
It depends on threshold parameters such as `q` and `eps`, which control how strictly cells are assigned to a branch.

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
q = 0.001  # Determines the threshold for selecting high-probability cells.
eps = 0.001  # A small tolerance subtracted from the threshold, allowing flexibility in the selection.

In [ ]:
masks = palantir.presults.select_branch_cells(
    adata,
    pseudo_time_key='palantir_pseudotime',
    fate_prob_key='palantir_fate_probabilities',
    q=q,
    eps=eps,
    masks_key='branch_masks',
)

In [ ]:
palantir.plot.plot_branch_selection(
    adata,
    pseudo_time_key='palantir_pseudotime',
    fate_prob_key='palantir_fate_probabilities',
    masks_key='branch_masks',
    fates=None,
    embedding_basis='X_umap',
    figsize=(15, 5),
)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Pseudotime
palantir.plot.plot_trajectories(
    adata,
    cell_color='palantir_pseudotime',
    pseudotime_interval=(0, 0.9),
    ax=axes[0],
    scanpy_kwargs={"cmap": "viridis"},
)
axes[0].set_title('Pseudotime', fontsize=16)

# Plot 2: Entropy
palantir.plot.plot_trajectories(
    adata,
    cell_color='palantir_entropy',
    pseudotime_interval=(0, 0.9),
    ax=axes[1],
    scanpy_kwargs={"cmap": "viridis"},
)
axes[1].set_title('Entropy', fontsize=16)

# Plot 3: Branch selection
palantir.plot.plot_trajectories(
    adata,
    cell_color='branch_selection',
    pseudotime_interval=(0, 0.9),
    ax=axes[2],
)
axes[2].set_title('Branch Selection', fontsize=16)

plt.tight_layout()

____

### 10.1 - Branch-QC
<hr style="border:2px solid black"> </hr>

**Branch QC – Table Interpretation**

| Metric | Description |
|------:|-----------:|
| n_cells | Number of cells in the branch mask. More cells usually mean more stable trends. |
| coverage | Fraction of pseudotime bins with ≥1 cell from this branch. Low values mean poor coverage. |
| pt_min | Minimum pseudotime value covered by this branch. |
| pt_max | Maximum pseudotime value covered by this branch. |
| keep (True) | Branch passes the minimum cell count and pseudotime coverage thresholds (recommended for stable trend analysis). |
| keep (False) | Branch fails one or more thresholds (may be underpowered or unevenly distributed along pseudotime). Interpret with caution. |


No data is removed automatically. The QC flag provides guidance for interpretation only.

In [ ]:
qc = tools.palantir_analysis.branch_qc(
    adata,
    masks_key="branch_masks",
    pseudotime_key="palantir_pseudotime",
    min_cells=150,
    n_bins=20,
    min_coverage=0.5,
)
qc

**Branch QC – Distribution Plots**

These plots show the pseudotime distribution for the largest branches
(sorted by number of cells). Each panel represents one branch.

+ Histogram → distribution of cells along pseudotime

+ n → number of cells assigned to the branch

+ keep → QC flag indicating whether the branch passes the filtering thresholds

+ Vertical lines (if shown) → minimum (pt_min) and maximum (pt_max) pseudotime values observed in the branch

These plots provide a visual validation of the QC metrics reported in the table,
helping to assess whether a branch has sufficient cell support and pseudotime coverage for downstream analysis.

In [ ]:
tools.palantir_analysis.plot_branch_qc_distributions(
    adata,
    qc,
    bins=40,
    ncols=3,
)
plt.show()

____

## 11 - Gene expression trends and Clustering 
<hr style="border:1px solid black"> </hr>

Palantir shows how each gene's activity changes along pseudotime using Gene Expression Trends. It first sorts the cells by pseudotime, smooths the gene expression, and then interpolates it onto a uniform grid: each gene gets exactly 500 evenly spaced pseudotime points. This creates smooth and comparable trend curves for all genes. The parameter `palantir.presults.PSEUDOTIME_RES = 500` sets this resolution higher values give smoother curves without changing the biological meaning. The output is a matrix per branch with size *genes × 500 pseudotime points*.

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# This is the global default parameter that sets how fine Palantir’s pseudotime grid is.
palantir.presults.PSEUDOTIME_RES = 500

In [ ]:
trends = palantir.presults.compute_gene_trends(
    adata,
    lineages=None,
    masks_key='branch_masks',
    expression_key="MAGIC_imputed_data",
    pseudo_time_key='palantir_pseudotime',
    gene_trend_key='gene_trends',
    save_as_df=True,
)

For each lineage (branch), gene expression trends are analyzed along pseudotime.

Genes with **flat trends** are removed first, because they do not change across pseudotime and are therefore not informative for downstream clustering.

The remaining genes are ranked by how dynamically their smoothed expression trends change across pseudotime. Two complementary measures are used:

- **Dynamic range**: the difference between the maximum and minimum expression value along pseudotime
- **Variance**: the overall variability of the trend curve across pseudotime

With the **union** strategy, the final gene set is built by combining top-ranked genes from both measures. Roughly half of the genes are taken from the dynamic-range ranking and the other half from the variance ranking. If the two rankings overlap, the remaining positions are filled using a combined ranking.

Only these selected dynamic genes are used for downstream clustering into branch-specific gene programs and for the subsequent interpretation of cluster-level trend patterns.

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# ============================================================
# 1) Gene program discovery (Palantir trend clustering)
# ============================================================
branch = None                     # None = all branches; or set e.g. "branchA" for a single branch
n_dynamic_genes = 1500            # dynamic genes selected per branch before clustering
gene_ranking_mode = "union"       # "union" | "range" | "var"
trend_knn_neighbors = 30          # kNN size for Palantir trend clustering
max_pseudotime_bins_heatmap = 120  # downsample heatmap bins if the grid is larger
plot_representative_genes = True  # compute + plot representative genes per cluster
representatives_per_cluster = 1   # number of representative genes per cluster
representatives_ncols = 3         # columns in representative gene panel grid
save = "fig"                      # filename prefix for saved figures (None disables saving)
save_excel = "palantir_gene_programs.xlsx"


# ============================================================
# 2) Early/Late ranking settings (per branch)
# ============================================================
quantile_cutoff = 0.25            # early/late window size (0.25 = first/last 25% of bins)
top_n = 5                         # number of genes shown for early and late panels
force_equal_ylim = True           # same y-limits for early/late panels (fair comparison)


# ============================================================
# 3) Heatmap display settings (representative genes)
# ============================================================
scaling = "z-score"               # "none" | "z-score" | "quantile" | "percent"
cmap = "RdBu_r"
vmin, vmax = -2, 2                # mainly meaningful for scaling="z-score"

In [ ]:
computed = tools.palantir_analysis.plot_branch_gene_programs(
    adata,
    branch=branch,
    n_dynamic_genes=n_dynamic_genes,
    gene_ranking_mode=gene_ranking_mode,
    trend_knn_neighbors=trend_knn_neighbors,
    max_pseudotime_bins_heatmap=max_pseudotime_bins_heatmap,
    plot_representative_genes=plot_representative_genes,
    representatives_per_cluster=representatives_per_cluster,
    representatives_ncols=representatives_ncols,
    save=f"01_{save}",
    save_excel=save_excel,
)

In [ ]:
first_branch = list(computed.keys())[0]

In [ ]:
# If representatives were not computed (plot_representative_genes=False),
rep_dict = computed[first_branch].get("cluster_representatives", {}) or {}

if not rep_dict:
    print("No cluster_representatives found. Set plot_representative_genes=True to compute them.")
else:
    # Flatten representative genes across clusters into one list
    gene_list = [gene for genes_in_cluster in rep_dict.values()
        for gene in genes_in_cluster
    ]

    print("Representative genes:", gene_list[:top_n])

    palantir.plot.plot_gene_trend_heatmaps(
        adata,
        genes=gene_list[:top_n],
        gene_trend_key="gene_trends",
        scaling=scaling,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
    )
    plt.show()

In [ ]:
out = tools.palantir_analysis.plot_branch_early_late_genes(
    adata,
    gene_trend_key="gene_trends",
    pseudotime_key="palantir_pseudotime",
    entropy_key="palantir_entropy",
    top_n=top_n,
    quantile_cutoff=quantile_cutoff,
    force_equal_ylim=force_equal_ylim,
    umap_ncols=4,
    branches=None,
    save="02_palantir_trend_panels",
)
plt.show()

___

In [ ]:
adata.obs.columns

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
specific_gen = 'LAMA2'  # Specifies the gene to be plotted.
color = "total_counts"  # from adata.obs.columns

In [ ]:
for branch in list(computed.keys()):
    print(f"Plotting {specific_gen} in Branch {branch}")
    palantir.plot.plot_trend(
        adata,
        branch,
        specific_gen,
        color=color,
        position_layer="MAGIC_imputed_data"
    )
    plt.show()

In [ ]:
palantir.plot.plot_gene_trends(adata, [specific_gen])
plt.show()

____

### 11.2 - Clustering 
<hr style="border:2px solid black"> </hr>

**Gene Trend Community Analysis**

Genes are grouped into communities based on similar expression patterns along Palantir pseudotime. The clustering is performed separately for each selected branch.

Increasing `n_neighbors` produces smoother, more global communities, while decreasing it results in more fine-grained clusters.

Two gene selection strategies are supported:

- **HVG-based:** Uses the top `n` highly variable genes to capture general dynamic patterns.
- **using adjusted p-values and |logFC| + |score| (z-score):** 

The analysis generates expression trend plots and Excel tables with cluster assignments for each branch.

In [ ]:
branches = adata.obsm['branch_masks'].columns.tolist()
print("Available branches:")
for i, b in enumerate(branches):
    print(f"Branch {i}: {b}")

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
top_n = 500              # Number of top dynamic genes to include in the clustering
n_neighbors = 30         # Number of nearest neighbors used for Leiden clustering

trend_branches = branches[0]

# Output Excel file names
excel_hvg = "gene_communities_hvg.xlsx"  # File to save HVG-based gene community results
excel_logfc = "gene_communities_logfc.xlsx"  # File to save log fold change–based gene community results

In [ ]:
# -----------------------
# HVG-based clustering
# -----------------------
selected_genes_hvg, communities_hvg = tools.palantir_analysis.analyze_top_genes_communities(
    adata=adata,
    trend_branches=trend_branches,
    top_n=top_n,
    n_neighbors=n_neighbors,
    save_excel=excel_hvg,
    gene_source="hvg",               # instead of "markers"
    store_key="hvg_run",
)

In [ ]:
# -----------------------
# Marker-based clustering
# -----------------------
selected_genes_marker_table, communities_marker_table = tools.palantir_analysis.analyze_top_genes_communities(
    adata=adata,
    trend_branches=trend_branches,
    gene_source="markers",  # instead of gene_source="hvg"
    marker_table=f"rank_feature_{clusters_to_analyze}_filtered",  # key in adata.uns
    top_n=top_n,
    n_neighbors=n_neighbors,
    save_excel=excel_logfc,
    store_key="markers_run",
)

____

## 12 - Saving adata
<hr style="border:2px solid black"> </hr>

In [ ]:
utils.adata.save_h5ad(adata, "anndata_palantir.h5ad")
settings.close_logfile()